# StormEngine V8 — staged model development

Stage 1 pretrains the mask-aware spatial Encoder and Decoder by reconstructing the current ERA5 grid from the final sparse input hour. It uses the frozen V7-B 390-point contract, 2010–2015 training, and 2016 validation. It does not read 2017 or the August 2026 operational evaluation week.

In [ ]:
from pathlib import Path
import json, subprocess, sys, yaml

here = Path.cwd().resolve()
REPO = here if (here / 'pyproject.toml').exists() else here.parent
assert (REPO / 'pyproject.toml').exists(), REPO

def run_live(command):
    print('Running:', ' '.join(map(str, command)), flush=True)
    process = subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
    code = process.wait()
    if code:
        raise subprocess.CalledProcessError(code, command)
    return code

print('Repository:', REPO)

## 1. Create the ignored Windows configuration
Change only `WINDOWS_ERA5_ROOT` if the data folder moved. The generated `.local.yaml` file is ignored by Git.

In [ ]:
WINDOWS_ERA5_ROOT = Path(r'D:\Documents\py_projects\StormEngine-DL\DownloadDate')
assert (WINDOWS_ERA5_ROOT / 'cache' / 'stormengine_2010_2017' / 'metadata.json').exists(), WINDOWS_ERA5_ROOT
local_config = REPO / 'configs' / 'v8_reconstruction_windows.local.yaml'
local_config.write_text(yaml.safe_dump({
    'extends': 'v8_reconstruction.yaml',
    'data': {'era5_root': str(WINDOWS_ERA5_ROOT)},
    'training': {'batch_size': 16, 'num_workers': 0},
}, sort_keys=False), encoding='utf-8')
print('ERA5:', WINDOWS_ERA5_ROOT)
print('Local config:', local_config)

## 2. Verify CUDA

In [ ]:
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('PyTorch:', torch.__version__)
print('Device:', DEVICE)
if DEVICE == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

## 3. Preflight
This verifies the real 390-point cache contract and one complete forward pass.

In [ ]:
run_live([sys.executable, '-u', str(REPO / 'scripts' / 'train_v8_reconstruction.py'), 'preflight', '--config', str(local_config), '--device', DEVICE])

## 4. Two-batch smoke run
This checks backward propagation and checkpoint writing. Its metrics are not scientific results.

In [ ]:
run_live([sys.executable, '-u', str(REPO / 'scripts' / 'train_v8_reconstruction.py'), 'smoke', '--config', str(local_config), '--device', DEVICE, '--output-dir', 'artifacts/v8_spatial_smoke'])

## 5. Five-epoch capped pilot
Inspect learning direction, speed, and GPU memory before enabling the full run.

In [ ]:
run_live([sys.executable, '-u', str(REPO / 'scripts' / 'train_v8_reconstruction.py'), 'pilot', '--config', str(local_config), '--device', DEVICE, '--output-dir', 'artifacts/v8_spatial_pilot'])

## 6. Full Stage-1 training
Leave this locked until the pilot loss is finite and decreasing. Model selection uses only 2016.

In [ ]:
RUN_FULL = False
if RUN_FULL:
    run_live([sys.executable, '-u', str(REPO / 'scripts' / 'train_v8_reconstruction.py'), 'train', '--config', str(local_config), '--device', DEVICE])
else:
    print('Full run locked. Set RUN_FULL=True only after inspecting the pilot.')

## 7. Inspect the selected summary
Use the pilot directory first; switch to `v8_spatial_pretraining` after the full run. MSL is reported separately because no same-semantics pressure input exists.

In [ ]:
RESULT_NAME = 'v8_spatial_pilot'
summary_name = 'pilot_summary.json' if RESULT_NAME.endswith('pilot') else 'train_summary.json'
summary_path = REPO / 'artifacts' / RESULT_NAME / summary_name
print(json.dumps(json.loads(summary_path.read_text(encoding='utf-8')), indent=2))